# AIS Ship Trajectory Analysis with GeoPandas
This notebook queries AIS vessel positions from a SQLite database and visualizes ship trajectories on interactive maps using GeoPandas.

## 1. Set the MMSI of the ship you would like to work with:

In [28]:
# 338839000 MMSI of the ship that goes in the carribean
# 369970910 MMSI of the ship that goes around in the san diego bay
# 338812000 MMSI of the ship that goes chillin in a Jacksonville Florida bay
# 366998000 MMSI of the ship going to Mobile Alabama
# 369970968 MMSI of the another ship chillin in the Jacksonville Florida bay
# 368011000 MMSI of the ship going down the coast of California
# 369970707 MMSI of the ship leaving the cuban bay
# 368869000 MMSI of the ship that went to go chill in the Jacksonville bay

SHIP_MMSI = 369970910

## 2. Import Required Libraries

In [29]:
import sys
from pathlib import Path

# Add actint to path for imports
actint_path = Path("/home/daxtonb/JFN-Groundtruth-Tools/actint/src")
if str(actint_path) not in sys.path:
    sys.path.insert(0, str(actint_path))

from actint.data_processing.query_database import query_ais_positions, query_vessels

## 3. Connect to Database and Query Ship Positions

In [30]:
# Query positions for a specific ship (configure as needed)
# For example: query_ais_positions({"MMSI": "123456789"}) or {"vessel_id": value}

SHIP_QUERY = {"MMSI": SHIP_MMSI}  # Change MMSI or column name as needed

try:
    ship_positions = query_ais_positions(SHIP_QUERY, sort=True)
    print(f"Retrieved {len(ship_positions)} positions")
    if ship_positions:
        print(f"Start position: {ship_positions[-1]}")
        print(f"End position: {ship_positions[0]}")
except Exception as e:
    print(f"Error querying database: {e}")
    ship_positions = []

2023-01-01T00:00:02.000000
Retrieved 1328 positions
Start position: (2, 369970910, '2023-01-01T00:00:02.000000', 32.60973, -117.39904, 5.2, 274.2, 281.0, 'USS MONTGOMERY', None, 'NMGY', 90.0, 0.0, None, None, None, 35.0, 'A', '2026-02-26 03:51:09')
End position: (10688, 369970910, '2023-01-01T23:59:36.000000', 32.8593, -117.86774, 4.6, 300.1, 306.0, 'USS MONTGOMERY', None, 'NMGY', 90.0, 3.0, None, None, None, 35.0, 'A', '2026-02-26 03:51:09')


## 4 Interactive map with markers (ipyleaflet) 

### This might take a while to load depending on how  many points you have

In [31]:
from ipyleaflet import Map, Marker, AwesomeIcon, Popup, Rectangle
from ipywidgets import HTML
import math

# Example structure:
# ship_positions = [
#     [ship_id, timestamp, speed, lat, lon],
#     ...
# ]

middle_point = ship_positions[len(ship_positions)//2]

map1 = Map(center=(middle_point[3], middle_point[4]), zoom=10)

# first and last positions
last = ship_positions[0]
first = ship_positions[-1]

start_icon = AwesomeIcon(name="play", marker_color="green", icon_color="white")
end_icon = AwesomeIcon(name="flag", marker_color="red", icon_color="white")

def create_popup(position):
    ship_id = position[1]
    timestamp = position[2]
    lat = position[3]
    lon = position[4]

    html = HTML(f"""
    <b>Ship MMSI:</b> {ship_id}<br>
    <b>Timestamp:</b> {timestamp}<br>
    <b>Latitude:</b> {lat}<br>
    <b>Longitude:</b> {lon}<br>
    """)

    return Popup(child=html, close_button=True, auto_close=False)

# Add first and last markers
start_marker = Marker(location=(first[3], first[4]), icon=start_icon)
start_marker.popup = create_popup(first)
map1.add(start_marker)

end_marker = Marker(location=(last[3], last[4]), icon=end_icon)
end_marker.popup = create_popup(last)
map1.add(end_marker)

# Add a clickable marker every 50 positions
for i, position in enumerate(ship_positions[1:-1], start=1):
    if i % 100 == 0:
        icon = AwesomeIcon(name="circle", marker_color="blue", icon_color="white")
        marker = Marker(location=(position[3], position[4]), icon=icon)
        marker.popup = create_popup(position)
        map1.add(marker)
    elif i%4 == 0:
        marker = Marker(location=(position[3], position[4]))
        map1.add(marker)

map1

Map(center=[32.68814, -117.63175], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…

## 5. Filter and Analyze Multiple Ship Trajectories

## Points used to determine where the ship is going

In [32]:
NUMBER_DETECTIONS=300

middle_point = ship_positions[300]

map2 = Map(center=(middle_point[3], middle_point[4]), zoom=10)

# first and last positions
first = ship_positions[NUMBER_DETECTIONS-1]
last = ship_positions[0]

start_icon = AwesomeIcon(name="play", marker_color="green", icon_color="white")
end_icon = AwesomeIcon(name="flag", marker_color="red", icon_color="white")

# Add first and last markers
start_marker = Marker(location=(first[3], first[4]), icon=start_icon)
start_marker.popup = create_popup(first)
map2.add(start_marker)

end_marker = Marker(location=(last[3], last[4]), icon=end_icon)
end_marker.popup = create_popup(last)
map2.add(end_marker)

# Add a clickable marker every 50 positions
print(len(ship_positions[1:NUMBER_DETECTIONS-1]))
for i, position in enumerate(ship_positions[1:NUMBER_DETECTIONS-1], start=1):
    if i % 100 == 0:
        icon = AwesomeIcon(name="circle", marker_color="blue", icon_color="white")
        marker = Marker(location=(position[3], position[4]), icon=icon)
        marker.popup = create_popup(position)
        map2.add(marker)
    else:
        marker = Marker(location=(position[3], position[4]))
        map2.add(marker)

map2

298


Map(center=[32.61974, -117.20902], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…

## A couple of helper functions

In [33]:
def vectorize(lat1, lon1, lat2, lon2):
    lat = lat2-lat1
    lon = lon2-lon1
    return (lat, lon)


def within_angle(a, b, tolerance=15):
    diff = abs((a - b + 180) % 360 - 180)
    return diff <= tolerance

## Code to determine what direction the ship is gonig

In [34]:
VECTOR_DISTANCE_RATIO = 0.9
DEGREE_THRESHOLD= 15
 #This is the number of detections used to calculate where a ship is going.
from actint.tools.utils.important_locations import *
from actint.data_processing.query_database import query_ais_positions
from datetime import datetime, timedelta
from actint.tools.utils.distance_calculation import calculate_bearing, haversine_distance_nm
from actint.tools.lat_lon_context import identify_maritime_region


def calculate_vector_and_distance_sum(ship_mmsi: str, number_detections=300, tracking_time=timedelta(hours=1)):
    positions = query_ais_positions({"mmsi": ship_mmsi}, sort=True)
    
    position1 = positions[number_detections-1]        #This is 300 or whatever the number_detections is away from the most rescent position
    print(position1)

    rescent_reversed_positions = reversed(positions[0:number_detections-1])
    total_vector = [0.0, 0.0]
    total_distance = 0

    for position2 in rescent_reversed_positions:
        print(position2[3], position2[4], position2[2])
        latlng = vectorize(position1[3], position1[4], position2[3], position2[4])
        total_vector[0] += latlng[0]
        total_vector[1] += latlng[1]
        total_distance += math.hypot(latlng[0], latlng[1])
        position1 = position2
    
    

    vector_distance_ratio = math.hypot(total_vector[0], total_vector[1])/total_distance
    print("Vector distance ratio", vector_distance_ratio)
    if(vector_distance_ratio > VECTOR_DISTANCE_RATIO):                                                       #Can use this to describe if the ship is going fast or slow
        print("The ship is going toward something")
        return (total_vector, total_distance)
    else:
        print("The ship is doing wierd stuff acting like a reet.")
        return (total_vector, total_distance)
    

        
(total_vector, total_distance) = calculate_vector_and_distance_sum(SHIP_MMSI, 300)

print(total_vector)

2023-01-01T00:00:02.000000
(8592, 369970910, '2023-01-01T18:14:41.000000', 32.6192, -117.21011, 3.9, 221.9, 242.0, 'USS MONTGOMERY', None, 'NMGY', 90.0, 0.0, None, None, None, 35.0, 'A', '2026-02-26 03:51:09')
32.61868 -117.21121 2023-01-01T18:15:45.000000
32.61821 -117.21218 2023-01-01T18:16:48.000000
32.61801 -117.21282 2023-01-01T18:17:52.000000
32.61798 -117.21323 2023-01-01T18:18:55.000000
32.61812 -117.21378 2023-01-01T18:19:58.000000
32.61888 -117.2159 2023-01-01T18:21:01.000000
32.6198 -117.21871 2023-01-01T18:22:02.000000
32.62093 -117.22211 2023-01-01T18:23:05.000000
32.62212 -117.22558 2023-01-01T18:24:09.000000
32.62332 -117.22906 2023-01-01T18:25:11.000000
32.62453 -117.23261 2023-01-01T18:26:15.000000
32.62571 -117.23617 2023-01-01T18:27:18.000000
32.62683 -117.23967 2023-01-01T18:28:21.000000
32.62855 -117.24245 2023-01-01T18:29:25.000000
32.6307 -117.24521 2023-01-01T18:30:29.000000
32.63282 -117.24793 2023-01-01T18:31:32.000000
32.63497 -117.25077 2023-01-01T18:32:35.0

In [35]:
current_position = ship_positions[0]

print(current_position)
map3 = Map(center=(current_position[3], current_position[4]), zoom=10)

# Add first and last markers
current_pos_marker = Marker(location=(current_position[3], current_position[4]))
current_pos_marker.popup = create_popup(current_position)
map3.add(current_pos_marker)

(dy, dx) = total_vector

start_lat = current_position[3]
start_lon = current_position[4]

end_lat = start_lat + dy
end_lon = start_lon + dx

from ipyleaflet import Polyline

vector_line = Polyline(
    locations=[(start_lat, start_lon), (end_lat, end_lon)],
    color="red",
    weight=3
)

map3.add(vector_line)

map3

(10688, 369970910, '2023-01-01T23:59:36.000000', 32.8593, -117.86774, 4.6, 300.1, 306.0, 'USS MONTGOMERY', None, 'NMGY', 90.0, 3.0, None, None, None, 35.0, 'A', '2026-02-26 03:51:09')


Map(center=[32.8593, -117.86774], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', …

## Code to determine how long a ship has been staying at a specific place (uses information from the previous code)

In [36]:
from geographiclib.geodesic import Geodesic

DISTANCE_THRESHOLD = 500 # Meters from where the ship has been
TIME_THRESHOLD = timedelta(minutes=20) # A ship is considered to be staying still if it has been within DISTANCE_THRESHOLD for TIME_THRESHOLD minutes
NUMBER_DETECTIONS_TRESHOLD = 5

ship_staying_still = False

geod = Geodesic.WGS84
latest_position = ship_positions[0]

latest_time = datetime.strptime(ship_positions[0][2], '%Y-%m-%dT%H:%M:%S.%f')

number_detections_still = 0
time_stayed_still = 0

for (num, ship_position) in enumerate(ship_positions[1:]):
    path = geod.Inverse(latest_position[3], latest_position[4], ship_position[3], ship_position[4])
    distance = path['s12']
    if(distance > DISTANCE_THRESHOLD):
        break
    if(num > NUMBER_DETECTIONS_TRESHOLD and not ship_staying_still):
        if (TIME_THRESHOLD < latest_time - datetime.strptime(ship_position[2], '%Y-%m-%dT%H:%M:%S.%f')):
            ship_staying_still = True #at this point all the conditions for the ship to be staying still are true, it hasn't moved more than DISTANCE_THRESHOLD, has been still for NUMBER_DETECTIONS_TRESHOLD, for at least TIME_THRESHOLD
    time_stayed_still = latest_time - datetime.strptime(ship_position[2], '%Y-%m-%dT%H:%M:%S.%f')
    number_detections_still += 1

if(ship_staying_still):
    total_seconds = int(time_stayed_still.total_seconds())
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    print(f"The ship has been still for {number_detections_still} detections and for a total time of {hours}h {minutes}m {seconds}s")
else:
    print("The ship is not currently staying still.")

The ship is not currently staying still.
